In [31]:
# Parameters
input_file = "/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_neutron_1203203_unopt_rPBE_magres.magres"


*This code uses papermill on terminal to change input_file. The input_file is the file path to the magres file*

*Shiva Agarwal*

*Apr 23 2025*

In [32]:
import os
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms

atoms = MagresAtoms.load_magres(input_file)


In [33]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [34]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [35]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [36]:
nucleus = 'N'      # nucleus for which parameters are wanted
atom_label = 0      # site for which parameters wanted

In [37]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

14N1 sigma:
 [[192.75169091  -0.76208595  -4.89477246]
 [ -3.58478683 179.83469525  -3.15318783]
 [ -5.15531002  -4.0966988  197.5227086 ]]

14N2 sigma:
 [[192.75169091   0.76208595   4.89477246]
 [  3.58478683 179.83469525  -3.15318783]
 [  5.15531002  -4.0966988  197.5227086 ]]

14N3 sigma:
 [[192.75169091  -0.76208595   4.89477246]
 [ -3.58478683 179.83469525   3.15318783]
 [  5.15531002   4.0966988  197.5227086 ]]

14N4 sigma:
 [[192.75169091   0.76208595  -4.89477246]
 [  3.58478683 179.83469525   3.15318783]
 [ -5.15531002   4.0966988  197.5227086 ]]



In [38]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

14N1 sigma:
 1.226026615559198

14N2 sigma:
 1.226026615559197

14N3 sigma:
 1.2260266155592257

14N4 sigma:
 1.2260266155592265



In [39]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

                                         
CS_total[:,:] = atoms.species(nucleus).ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species(nucleus)[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = 0.0204 #electric quadrupole moment for 14N in barn
V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.379 -0.254 -0.934]
 [-0.254 -0.38   0.258]
 [-0.934  0.258  0.001]]

CS Tensor:
 [[192.752  -0.762  -4.895]
 [ -3.585 179.835  -3.153]
 [ -5.155  -4.097 197.523]]

CS isotropic Tensor:
 [[190.036   0.      0.   ]
 [  0.    190.036   0.   ]
 [  0.      0.    190.036]]

CS symmetric Tensor:
 [[192.752  -2.173  -5.025]
 [ -2.173 179.835  -3.625]
 [ -5.025  -3.625 197.523]]

CS antisymmetric Tensor:
 [[ 0.     1.411  0.13 ]
 [-1.411  0.     0.472]
 [-0.13  -0.472  0.   ]]


In [40]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 1.22362693 -0.76771123 -0.4559157 ] 

 Unsorted Eigenvectors:
 [[-0.75266222  0.61024134 -0.24719443]
 [ 0.2190674  -0.12194381 -0.96805949]
 [ 0.62089376  0.78277405  0.04190149]] 

Sorted Eigenvalues: 
 [-0.4559157  -0.76771123  1.22362693] 

Sorted Eigenvectors: 
 [[-0.24719443  0.61024134 -0.75266222]
 [-0.96805949 -0.12194381  0.2190674 ]
 [ 0.04190149  0.78277405  0.62089376]] 


For CS tensor
 Unsorted Eigenvalues:
 [200.88263262 190.82854582 178.39791632] 

 Unsorted Eigenvectors:
 [[ 0.50468222 -0.83306505 -0.22649169]
 [ 0.09565165  0.31469744 -0.94436025]
 [-0.85798987 -0.45493752 -0.23850625]] 

Sorted Eigenvalues: 
 [190.82854582 200.88263262 178.39791632] 

Sorted Eigenvectors: 
 [[-0.83306505  0.50468222 -0.22649169]
 [ 0.31469744  0.09565165 -0.94436025]
 [-0.45493752 -0.85798987 -0.23850625]] 



In [41]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.45591569948462657 -0.767711227533294 1.2236269270179339
CSA Tensor Components δyy, δxx, δzz: 
 190.82854582313524 200.88263261840441 178.39791632291798


In [42]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]

table_string = tabulate(table, headers=['Quantity', 'Value'], tablefmt='grid')
print(f'Parameters for {atoms.species(nucleus)[atom_label]}: \n', table_string)

Parameters for 14N1: 
 +--------------+------------+
| Quantity     |      Value |
+==============+============+
| CQ (MHz)     |   1.22363  |
+--------------+------------+
| etaq         |   0.254813 |
+--------------+------------+
| iso_cs (ppm) | 190.036    |
+--------------+------------+
| csa (ppm)    | -11.6384   |
+--------------+------------+
| etas         |   0.863868 |
+--------------+------------+


In [43]:

# Derive output name from input file
base_name = os.path.splitext(os.path.basename(input_file))[0]
output_txt = "/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/N14_all_results.txt"

# Save to .txt file
with open(output_txt, 'a') as f:
    f.write(f"\n\n===== Results for: {base_name} =====\n\n")
    f.write(table_string)
    f.write("\n")

print(f"Saved table to {output_txt}")

Saved table to /home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/N14_all_results.txt


In [44]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[ 0.61024134 -0.24719443 -0.75266222]
 [-0.12194381 -0.96805949  0.2190674 ]
 [ 0.78277405  0.04190149  0.62089376]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
3.064088843670231 51.61856927996166 16.22799217696503 

Direction cosine csa: 

[[ 0.50468222 -0.83306505 -0.22649169]
 [ 0.09565165  0.31469744 -0.94436025]
 [-0.85798987 -0.45493752 -0.23850625]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
27.934166346012645 103.79839470400408 -76.51314577193781 



In [45]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -24.8794669020227 chi: 100.63162404579924 xi: 77.67739913151136 

